In [3]:
!pip install requests beautifulsoup4 pandas

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

base_url = "https://books.toscrape.com/catalogue/page-{}.html"

books = []

for page in range(1, 51):

    url = base_url.format(page)
    response = requests.get(url)

    if response.status_code == 200:

        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):

            # Basic information
            title = book.h3.a["title"]
            price = book.select_one(".price_color").text.strip()
            availability = book.select_one(".availability").text.strip()
            rating = book.select_one(".star-rating")["class"][1]

            # Book detail page URL
            book_url = "https://books.toscrape.com/catalogue/" + book.h3.a["href"]

            # Open individual book page
            book_response = requests.get(book_url)
            book_soup = BeautifulSoup(book_response.text, "html.parser")

            # Category
            category = book_soup.select("ul.breadcrumb li a")[-1].text.strip()

            # Number of reviews
            table = book_soup.select_one("table.table-striped")
            rows = table.select("tr")

            num_reviews = None

            for row in rows:
                cells = row.select("td")

                if len(cells) == 2 and cells[0].text.strip() == "Number of reviews":
                    num_reviews = cells[1].text.strip()

            books.append({
                "Name": title,
                "Price": price,
                "Rating": rating,
                "Availability": availability,
                "Category": category,
                "Number of Reviews": num_reviews,
                "URL": book_url
            })

        print(f"Page {page} scraped successfully")

    else:
        print(f"Error on page {page}: {response.status_code}")

    time.sleep(0.5)


df = pd.DataFrame(books)

print(df.head())
print(df.shape)

Page 1 scraped successfully
Page 2 scraped successfully
Page 3 scraped successfully
Page 4 scraped successfully
Page 5 scraped successfully
Page 6 scraped successfully


KeyboardInterrupt: 

In [5]:
df

,Name,Price,Rating,Availability,Category,Number of Reviews,URL
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,None,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,None,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,Fiction,None,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,None,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,None,https://books.toscrape.com/catalogue/sapiens-a...
...,...,...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,Â£55.53,One,In stock,Classics,None,https://books.toscrape.com/catalogue/alice-in-...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Â£57.06,Four,In stock,Sequential Art,None,https://books.toscrape.com/catalogue/ajin-demi...
997,A Spy's Devotion (The Regency Spies of London #1),Â£16.97,Five,In stock,Historical Fiction,None,https://books.toscrape.com/catalogue/a-spys-de...
998,1st to Die (Women's Murder Club #1),Â£53.98,One,In stock,Mystery,None,https://books.toscrape.com/catalogue/1st-to-di...


In [6]:
df.to_csv("books_data.csv", index=False)

In [5]:
df = pd.read_csv("C:\\Users\\bhoom\\OneDrive\\Desktop\\book_scraping\\books_data.csv")

In [6]:
df.columns

Index(['Name', 'Price', 'Rating', 'Availability', 'Category',
       'Number of Reviews', 'URL'],
      dtype='str')

In [8]:
# Name

df["Name"].isna().sum()

np.int64(0)

In [9]:
df["Name"].duplicated().sum()

np.int64(1)

In [9]:
# Price

df["Price"].isna().sum()

np.int64(0)

In [11]:
df["Price"].dtypes

<StringDtype(storage='python', na_value=nan)>

In [12]:
df["Price"].unique()

<StringArray>
['Â£51.77', 'Â£53.74', 'Â£50.10', 'Â£47.82', 'Â£54.23', 'Â£22.65', 'Â£33.34',
 'Â£17.93', 'Â£22.60', 'Â£52.15',
 ...
 'Â£41.24', 'Â£39.07', 'Â£29.82', 'Â£37.26', 'Â£20.30', 'Â£34.65', 'Â£43.38',
 'Â£57.06', 'Â£16.97', 'Â£26.08']
Length: 903, dtype: str

In [13]:
df["Price"] = (
    df["Price"]
    .str.replace("Â£", "", regex=False)
    .str.strip()
    .astype(float)
)

In [14]:
df["Price"].unique()

array([51.77, 53.74, 50.1 , 47.82, 54.23, 22.65, 33.34, 17.93, 22.6 ,
       52.15, 13.99, 20.66, 17.46, 52.29, 35.02, 57.25, 23.88, 37.59,
       51.33, 45.17, 12.84, 37.32, 30.52, 25.27, 34.53, 54.64, 22.5 ,
       53.13, 40.3 , 44.18, 17.66, 31.05, 23.82, 36.89, 15.94, 33.29,
       18.02, 19.63, 52.22, 33.63, 57.31, 26.41, 47.61, 23.11, 45.07,
       31.77, 50.27, 14.27, 18.78, 25.52, 16.28, 31.12, 19.49, 17.27,
       19.09, 56.13, 56.41, 56.5 , 45.22, 38.16, 54.11, 42.96, 23.89,
       16.77, 20.59, 37.13, 56.06, 58.11, 49.05, 40.76, 19.73, 32.24,
       41.83, 39.58, 39.25, 25.02, 51.04, 19.83, 50.4 , 13.61, 13.34,
       18.97, 36.28, 10.16, 15.44, 48.41, 46.35, 14.07, 14.86, 33.37,
       56.4 , 14.02, 46.91, 45.61, 19.92, 40.11, 53.9 , 35.67, 22.  ,
       57.36, 29.17, 54.63, 46.03, 33.97, 22.11, 29.69, 15.97, 21.96,
       54.35, 37.97, 51.99, 43.29, 36.72, 17.08, 29.14, 28.81, 49.46,
       37.92, 28.09, 30.81, 42.95, 56.76, 16.64, 55.53, 28.13, 52.37,
       54.  , 21.87,

In [ ]:
df.rename(columns={"Price": "Price in £"}, inplace=True)

In [16]:
df

,Name,Price in £,Rating,Availability,Category,Number of Reviews,URL
0,A Light in the Attic,51.77,Three,In stock,Poetry,NaN,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,Historical Fiction,NaN,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,Fiction,NaN,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,Mystery,NaN,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,History,NaN,https://books.toscrape.com/catalogue/sapiens-a...
...,...,...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,One,In stock,Classics,NaN,https://books.toscrape.com/catalogue/alice-in-...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,Four,In stock,Sequential Art,NaN,https://books.toscrape.com/catalogue/ajin-demi...
997,A Spy's Devotion (The Regency Spies of London #1),16.97,Five,In stock,Historical Fiction,NaN,https://books.toscrape.com/catalogue/a-spys-de...
998,1st to Die (Women's Murder Club #1),53.98,One,In stock,Mystery,NaN,https://books.toscrape.com/catalogue/1st-to-di...


In [17]:
df["Price in £"].isna().sum()

np.int64(0)

In [18]:
df["Price in £"].duplicated().sum()

np.int64(97)

In [19]:
df["Price in £"].dtypes

dtype('float64')

In [20]:
# Rating

df["Rating"].isna().sum()

np.int64(0)

In [22]:
df["Rating"].duplicated().sum()

np.int64(995)

In [23]:
df["Rating"].unique()

<StringArray>
['Three', 'One', 'Four', 'Five', 'Two']
Length: 5, dtype: str

In [27]:
df["Rating"] = df["Rating"].replace({
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
})
df["Rating"] = pd.to_numeric(df["Rating"])

In [28]:
df["Rating"].unique()

array([3, 1, 4, 5, 2])

In [29]:
df["Rating"].dtypes

dtype('int64')

In [30]:
# Availability

df["Availability"].isna().sum()

np.int64(0)

In [31]:
df["Availability"].duplicated().sum()

np.int64(999)

In [32]:
df["Availability"].unique()

<StringArray>
['In stock']
Length: 1, dtype: str

In [33]:
# Category

df["Category"].isna().sum()

np.int64(0)

In [34]:
df["Category"].duplicated().sum()

np.int64(950)

In [35]:
df["Category"].unique()

<StringArray>
[            'Poetry', 'Historical Fiction',            'Fiction',
            'Mystery',            'History',        'Young Adult',
           'Business',            'Default',     'Sequential Art',
              'Music',    'Science Fiction',           'Politics',
             'Travel',           'Thriller',     'Food and Drink',
            'Romance',          'Childrens',         'Nonfiction',
                'Art',       'Spirituality',         'Philosophy',
          'New Adult',       'Contemporary',            'Fantasy',
      'Add a comment',            'Science',             'Health',
             'Horror',          'Self Help',           'Religion',
          'Christian',              'Crime',      'Autobiography',
  'Christian Fiction',          'Biography',     'Womens Fiction',
            'Erotica',           'Cultural',         'Psychology',
              'Humor',         'Historical',             'Novels',
      'Short Stories',           'Suspense',    

In [37]:
# Number of Reviews	

df["Number of Reviews"].isna().sum()

np.int64(1000)

In [38]:
df["Number of Reviews"]=df["Number of Reviews"].fillna(0)

In [39]:
df["Number of Reviews"].isna().sum()

np.int64(0)

In [ ]:
# URL

df["URL"].isna().sum()

np.int64(0)

In [41]:
df["URL"].duplicated().sum()

np.int64(0)

In [42]:
df["URL"].dtypes

<StringDtype(storage='python', na_value=nan)>

In [43]:
df

,Name,Price in £,Rating,Availability,Category,Number of Reviews,URL
0,A Light in the Attic,51.77,3,In stock,Poetry,0.0,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,Historical Fiction,0.0,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,Fiction,0.0,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,Mystery,0.0,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,History,0.0,https://books.toscrape.com/catalogue/sapiens-a...
...,...,...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,1,In stock,Classics,0.0,https://books.toscrape.com/catalogue/alice-in-...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,4,In stock,Sequential Art,0.0,https://books.toscrape.com/catalogue/ajin-demi...
997,A Spy's Devotion (The Regency Spies of London #1),16.97,5,In stock,Historical Fiction,0.0,https://books.toscrape.com/catalogue/a-spys-de...
998,1st to Die (Women's Murder Club #1),53.98,1,In stock,Mystery,0.0,https://books.toscrape.com/catalogue/1st-to-di...


In [44]:
df.to_csv("Books_online_cleaned_dataset.csv", index=False)